# Model Training and Verification Pipeline

This notebook defines utility functions, dataset classes, the embedding model, training loop, embedding extraction, verification pair sampling, and the main training/evaluation pipeline.

In [56]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def normalize_embeddings(x):
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    return x / (norm + 1e-12)


def cosine_similarity(a, b):
    return float(np.dot(a, b))


def compute_eer(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return float(eer)

In [57]:
class EarDataset(Dataset):
    def __init__(self, df, images_root, transform=None, label_column="label"):
        self.df = df.reset_index(drop=True)
        self.images_root = images_root
        self.transform = transform
        self.label_column = label_column

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.images_root, str(row["image_path"]))
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label = int(row[self.label_column])
        return image, label

In [58]:
class EarDataset(Dataset):
    def __init__(self, df, images_root, transform=None, label_column="label"):
        self.df = df.reset_index(drop=True)
        self.images_root = images_root
        self.transform = transform
        self.label_column = label_column

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.images_root, str(row["image_path"]))
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label = int(row[self.label_column])
        return image, label

In [59]:
class EmbeddingClassifier(nn.Module):
    def __init__(self, num_classes, embedding_dim=512):
        super().__init__()

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.embedding = nn.Sequential(
            nn.Linear(in_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        emb = self.embedding(features)
        logits = self.classifier(emb)
        return logits, emb

In [60]:
def batch_accuracy(logits, y):
    pred = logits.argmax(dim=1)
    acc = (pred == y).float().mean().item()
    return acc


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    all_losses = []
    all_accs = []

    for x, y in tqdm(loader, desc="train", leave=False):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        all_losses.append(loss.item())
        all_accs.append(batch_accuracy(logits, y))

    return np.mean(all_losses), np.mean(all_accs)


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    all_losses = []
    all_accs = []

    for x, y in tqdm(loader, desc="val", leave=False):
        x = x.to(device)
        y = y.to(device)

        logits, _ = model(x)
        loss = criterion(logits, y)

        all_losses.append(loss.item())
        all_accs.append(batch_accuracy(logits, y))

    return np.mean(all_losses), np.mean(all_accs)

In [61]:
@torch.no_grad()
def extract_embeddings(model, df, images_root, img_size, batch_size, num_workers, device):
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    temp_df = df.copy()
    temp_df["label"] = 0

    dataset = EarDataset(temp_df, images_root, transform=transform, label_column="label")
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    embeddings = []

    for x, _ in tqdm(loader, desc="embeddings", leave=False):
        x = x.to(device)
        _, emb = model(x)
        embeddings.append(emb.cpu().numpy())

    embeddings = np.vstack(embeddings)
    embeddings = normalize_embeddings(embeddings)

    subject_ids = temp_df["subject_id"].astype(str).values
    return embeddings, subject_ids

In [62]:
def sample_verification_pairs(embeddings, subject_ids, n_genuine, n_impostor, seed=42):
    rng = np.random.RandomState(seed)

    indices_by_subject = {}
    for i, s in enumerate(subject_ids):
        if s not in indices_by_subject:
            indices_by_subject[s] = []
        indices_by_subject[s].append(i)

    subjects = list(indices_by_subject.keys())

    genuine_scores = []
    impostor_scores = []

    valid_subjects = [s for s in subjects if len(indices_by_subject[s]) >= 2]

    for _ in range(n_genuine):
        s = rng.choice(valid_subjects)
        i1, i2 = rng.choice(indices_by_subject[s], size=2, replace=False)
        score = cosine_similarity(embeddings[i1], embeddings[i2])
        genuine_scores.append(score)

    for _ in range(n_impostor):
        s1, s2 = rng.choice(subjects, size=2, replace=False)
        i1 = rng.choice(indices_by_subject[s1])
        i2 = rng.choice(indices_by_subject[s2])
        score = cosine_similarity(embeddings[i1], embeddings[i2])
        impostor_scores.append(score)

    y_true = np.array([1] * len(genuine_scores) + [0] * len(impostor_scores))
    y_score = np.array(genuine_scores + impostor_scores)

    return y_true, y_score


def rank1_score(embeddings, subject_ids):
    sims = embeddings @ embeddings.T
    np.fill_diagonal(sims, -np.inf)

    nn_idx = sims.argmax(axis=1)
    pred_subjects = subject_ids[nn_idx]

    acc = np.mean(pred_subjects == subject_ids)
    return float(acc)

In [63]:
def filter_by_datasets(df, datasets):
    return df[df["dataset"].isin(datasets)].copy()


def split_train_val_by_subject(df, val_ratio=0.1, seed=42):
    subjects = df["subject_id"].astype(str).unique().tolist()

    rng = np.random.RandomState(seed)
    rng.shuffle(subjects)

    n_val = max(1, int(len(subjects) * val_ratio))
    val_subjects = set(subjects[:n_val])
    train_subjects = set(subjects[n_val:])

    df_train = df[df["subject_id"].astype(str).isin(train_subjects)].copy()
    df_val = df[df["subject_id"].astype(str).isin(val_subjects)].copy()

    return df_train, df_val

In [64]:
def main():
    set_seed(SEED)
    make_dir(OUTPUT_DIR)

    device = get_device()
    print("Device:", device)

    df = pd.read_csv(METADATA_CSV)

    # -----------------------------
    # TRAIN
    # -----------------------------
    df_train_all = df[df["age_group"] == TRAIN_AGE_GROUP].copy()
    df_train_all = filter_by_datasets(df_train_all, TRAIN_DATASETS)

    if len(df_train_all) == 0:
        raise ValueError("No hay datos de entrenamiento después del filtrado.")

    # labels por sujeto
    subjects = sorted(df_train_all["subject_id"].astype(str).unique())
    subject_to_label = {s: i for i, s in enumerate(subjects)}
    df_train_all["label"] = df_train_all["subject_id"].astype(str).map(subject_to_label)

    df_train, df_val = split_train_val_by_subject(df_train_all, VAL_RATIO, SEED)

    print("Train datasets:", TRAIN_DATASETS)
    print("Test datasets:", TEST_DATASETS)
    print("Train subjects:", df_train_all["subject_id"].nunique())
    print("Train images:", len(df_train))
    print("Val images:", len(df_val))

    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    train_dataset = EarDataset(df_train, IMAGES_ROOT, transform=train_transform, label_column="label")
    val_dataset = EarDataset(df_val, IMAGES_ROOT, transform=val_transform, label_column="label")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    model = EmbeddingClassifier(num_classes=len(subjects), embedding_dim=512)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    best_val_acc = -1
    best_model_path = os.path.join(OUTPUT_DIR, "best_model.pt")

    history = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    }

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        print(f"Epoch {epoch+1:02d} | train loss {train_loss:.4f} acc {train_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print("  ✔ Modelo guardado en:", best_model_path)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(float(train_loss))
        history["train_acc"].append(float(train_acc))
        history["val_loss"].append(float(val_loss))
        history["val_acc"].append(float(val_acc))

    print("Training finished. Best val acc:", best_val_acc)




    # -----------------------------
    # TEST
    # -----------------------------
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()

    df_test = df[df["age_group"] == TEST_AGE_GROUP].copy()
    df_test = filter_by_datasets(df_test, TEST_DATASETS)

    if len(df_test) == 0:
        raise ValueError("No hay datos de test después del filtrado.")

    print("Test images:", len(df_test))
    print("Test subjects:", df_test["subject_id"].nunique())

    test_embeddings, test_subject_ids = extract_embeddings(
        model,
        df_test,
        IMAGES_ROOT,
        IMG_SIZE,
        BATCH_SIZE,
        NUM_WORKERS,
        device
    )

    y_true, y_score = sample_verification_pairs(
        test_embeddings,
        test_subject_ids,
        N_GENUINE_PAIRS,
        N_IMPOSTOR_PAIRS,
        SEED
    )

    auc = roc_auc_score(y_true, y_score)
    eer = compute_eer(y_true, y_score)
    rank1 = rank1_score(test_embeddings, test_subject_ids)

    print("\n===== RESULTADOS =====")
    print("ROC-AUC:", round(auc, 4))
    print("EER:", round(eer, 4))
    print("Rank-1:", round(rank1, 4))

    results_path = os.path.join(OUTPUT_DIR, "results.txt")
    with open(results_path, "w", encoding="utf-8") as f:
        f.write("Train datasets: " + str(TRAIN_DATASETS) + "\n")
        f.write("Test datasets: " + str(TEST_DATASETS) + "\n")
        f.write("ROC-AUC: " + str(round(auc, 6)) + "\n")
        f.write("EER: " + str(round(eer, 6)) + "\n")
        f.write("Rank-1: " + str(round(rank1, 6)) + "\n")


    print("Resultados guardados en:", results_path)

    history_df = pd.DataFrame(history)
    history_path = os.path.join(OUTPUT_DIR, "training_history.csv")
    history_df.to_csv(history_path, index=False)
    print("Historial guardado en:", history_path)

    experiment_result = {
    "experiment_name": EXPERIMENT_NAME,
    "train_datasets": "+".join(TRAIN_DATASETS),
    "test_datasets": "+".join(TEST_DATASETS),
    "train_age_group": TRAIN_AGE_GROUP,
    "test_age_group": TEST_AGE_GROUP,
    "roc_auc": float(auc),
    "eer": float(eer),
    "rank1": float(rank1),
    "num_train_images": int(len(df_train)),
    "num_val_images": int(len(df_val)),
    "num_test_images": int(len(df_test)),
    "num_train_subjects": int(df_train_all["subject_id"].nunique()),
    "num_test_subjects": int(df_test["subject_id"].nunique()),
    }

    results_csv = "all_experiments_results.csv"

    if os.path.exists(results_csv):
        results_df = pd.read_csv(results_csv)
        results_df = pd.concat([results_df, pd.DataFrame([experiment_result])], ignore_index=True)
    else:
        results_df = pd.DataFrame([experiment_result])

    results_df.to_csv(results_csv, index=False)
    print("Resumen global guardado en:", results_csv)

    return history_df



if __name__ == "__main__":
    history_df = main()

Device: cuda
Train datasets: ['AMI', 'BIPLab', 'UERC']
Test datasets: ['EICZA']
Train subjects: 366
Train images: 2831
Val images: 473


Epoch 01 | train loss 5.4227 acc 0.0975 | val loss 6.2807 acc 0.0000
  ✔ Modelo guardado en: runs/ALL_to_EICZA/best_model.pt


Epoch 02 | train loss 3.5902 acc 0.4959 | val loss 6.1150 acc 0.0000


Epoch 03 | train loss 2.3963 acc 0.8063 | val loss 6.2368 acc 0.0000


Epoch 04 | train loss 1.4799 acc 0.9712 | val loss 6.2206 acc 0.0000


...

Epoch 07 | train loss 0.2224 acc 1.0000 | val loss 6.2975 acc 0.0000


Epoch 08 | train loss 0.1476 acc 1.0000 | val loss 6.3126 acc 0.0000


Epoch 09 | train loss 0.1072 acc 1.0000 | val loss 6.3199 acc 0.0000


Epoch 10 | train loss 0.0807 acc 1.0000 | val loss 6.3141 acc 0.0000
Training finished. Best val acc: 0.0


ValueError: No hay datos de test después del filtrado.